In [1]:
# 실행 시간 측정: 단일 AEDTPLT export
import time

start_aedtplt = time.perf_counter()

# 첫 번째 variation만 추출
test_table_1_aedtplt = ParametricTable.iloc[:1]

print("=" * 70)
print(f"⏱️  AEDTPLT Export 시간 측정 (1개 Variation)")
print("=" * 70)

# AEDTPLT export 실행
aedtplt_files_test = export_aedtplt_for_variations(
    m2d_obj=m2d,
    parametric_table=test_table_1_aedtplt,
    quantity="Mag_B",
    solution="Setup1 : Transient",
    assignment="AllObjects",
    output_dir=r"D:\KDHe10\e10_example\AEDTPLT_Exports_Test",
    intrinsics={"Time": "0.06s"}
)

elapsed_aedtplt = time.perf_counter() - start_aedtplt

print(f"\n{'='*70}")
print("📊 AEDTPLT 성능 측정 결과")
print(f"{'='*70}")
print(f"✅ 총 소요 시간: {elapsed_aedtplt:.3f} 초")
print(f"📦 생성된 파일: {len(aedtplt_files_test)}개")
print(f"\n💡 전체 {len(ParametricTable)}개 variation 예상 시간:")
print(f"   약 {elapsed_aedtplt * len(ParametricTable):.1f} 초 ({elapsed_aedtplt * len(ParametricTable) / 60:.1f} 분)")
print(f"{'='*70}")

NameError: name 'ParametricTable' is not defined

# Maxwell 2D Field Data Export

## Setup: AEDT Connection & Utilities

1. Mesh는 case로 내보내기
2. fld로 field내보내기

In [23]:
import ansys.aedt.core
import os

import tempfile
import time
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.
from ansys.aedt.core import Desktop

import ansys.aedt.core
from ansys.aedt.core import Desktop
import subprocess
import psutil
import time
import os
import re

def get_aedt_processes_detailed():
    """
    실행 중인 AEDT 프로세스를 상세히 확인합니다.
    
    Returns:
    --------
    list : AEDT 프로세스 정보 리스트
    """
    print("🔍 실행 중인 AEDT 프로세스 검색...")
    
    aedt_processes = []
    try:
        for proc in psutil.process_iter(['pid', 'name', 'cmdline', 'create_time']):
            try:
                pinfo = proc.info
                process_name = pinfo['name'] if pinfo['name'] else ""
                
                # AEDT 관련 프로세스 필터링
                if any(keyword in process_name.lower() for keyword in ['ansysedt', 'aedt']):
                    # 포트 정보 추출 시도
                    ports = []
                    try:
                        connections = proc.connections()
                        for conn in connections:
                            if conn.status == 'LISTEN':
                                ports.append(conn.laddr.port)
                    except (psutil.AccessDenied, psutil.NoSuchProcess):
                        pass
                    
                    aedt_processes.append({
                        'pid': pinfo['pid'],
                        'name': process_name,
                        'cmdline': pinfo['cmdline'] if pinfo['cmdline'] else [],
                        'create_time': time.ctime(pinfo['create_time']),
                        'ports': ports
                    })
                    
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
    
        if aedt_processes:
            print(f"✅ {len(aedt_processes)}개의 AEDT 프로세스 발견:")
            for i, proc in enumerate(aedt_processes):
                print(f"\n📋 프로세스 {i+1}:")
                print(f"   PID: {proc['pid']}")
                print(f"   이름: {proc['name']}")
                print(f"   생성시간: {proc['create_time']}")
                if proc['ports']:
                    print(f"   열린 포트: {proc['ports']}")
                else:
                    print(f"   열린 포트: 없음")
        else:
            print("❌ AEDT 프로세스가 없습니다.")
            
        return aedt_processes
        
    except Exception as e:
        print(f"❌ 프로세스 검색 중 오류: {e}")
        return []

def try_connect_to_existing_desktop():
    """
    기존 AEDT Desktop에 연결을 시도합니다.
    
    Returns:
    --------
    Desktop or None : 연결된 Desktop 객체 또는 None
    """
    print("🔗 기존 AEDT Desktop 연결 시도...")
    
    try:
        # 방법 1: new_desktop_session=False로 기존 세션에 연결
        desktop = Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=False,
            non_graphical=NG_MODE
        )
        print("✅ 기존 AEDT Desktop에 성공적으로 연결되었습니다!")
        return desktop
        
    except Exception as e:
        print(f"❌ 기존 Desktop 연결 실패: {e}")
        return None

def try_connect_with_ports(port_list):
    """
    특정 포트들을 시도해서 AEDT에 연결합니다.
    
    Parameters:
    -----------
    port_list : list
        시도할 포트 번호 리스트
        
    Returns:
    --------
    Desktop or None : 연결된 Desktop 객체 또는 None
    """
    AEDT_VERSION='251'
    NG_MODE=False
    for port in port_list:
        try:
            print(f"🔗 포트 {port}로 연결 시도...")
            desktop = Desktop(
                specified_version=AEDT_VERSION,
                new_desktop_session=False,
                port=port,
                non_graphical=NG_MODE
            )
            print(f"✅ 포트 {port}로 성공적으로 연결되었습니다!")
            return desktop
        except Exception as e:
            print(f"❌ 포트 {port} 연결 실패: {e}")
            continue
    
    return None

def get_desktop_connection():
    """
    다양한 방법으로 AEDT Desktop 연결을 시도합니다.
    
    Returns:
    --------
    Desktop : 연결된 Desktop 객체
    """
    print("=" * 60)
    print("🎯 AEDT Desktop 연결 시도")
    print("=" * 60)
    
    # 1. 기존 Desktop 연결 시도
    desktop = try_connect_to_existing_desktop()
    if desktop:
        return desktop
    
    # 2. 프로세스에서 포트 찾아서 연결 시도
    processes = get_aedt_processes_detailed()
    all_ports = []
    
    for proc in processes:
        all_ports.extend(proc['ports'])
    
    if all_ports:
        desktop = try_connect_with_ports(all_ports)
        if desktop:
            return desktop
    
    # 3. 일반적인 AEDT 포트들 시도
    common_ports = [56800, 56801, 56802, 56803, 56804, 56805]
    print("\n🔍 일반적인 AEDT 포트들 시도...")
    desktop = try_connect_with_ports(common_ports)
    if desktop:
        return desktop
    
    # 4. 새로운 Desktop 세션 생성
    print("\n🆕 새로운 AEDT Desktop 세션을 생성합니다...")
    try:
        desktop = Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=True,
            non_graphical=NG_MODE
        )
        print("✅ 새로운 AEDT Desktop이 생성되었습니다!")
        return desktop
    except Exception as e:
        print(f"❌ 새 Desktop 생성 실패: {e}")
        return None

def check_current_desktop_status(desktop):
    """
    현재 Desktop의 상태를 확인합니다.
    
    Parameters:
    -----------
    desktop : Desktop
        확인할 Desktop 객체
    """
    if not desktop:
        print("❌ Desktop 객체가 없습니다.")
        return
    
    try:
        print("\n" + "=" * 40)
        print("📊 현재 Desktop 상태:")
        print("=" * 40)
        
        # 기본 정보
        print(f"AEDT 버전: {desktop.aedt_version_id}")
        print(f"프로세스 ID: {desktop.aedt_process_id}")
        
        # 프로젝트 정보
        try:
            projects = desktop.project_list()
            print(f"\n📁 열린 프로젝트 ({len(projects)}개):")
            for i, proj_name in enumerate(projects, 1):
                print(f"  {i}. {proj_name}")
            
            # 활성 프로젝트
            active_proj = desktop.active_project()
            if active_proj:
                proj_name = active_proj.GetName()
                print(f"\n🎯 활성 프로젝트: {proj_name}")
                
                # 디자인 목록
                try:
                    design_list = active_proj.GetTopDesignList()
                    print(f"📐 디자인 ({len(design_list)}개):")
                    for i, design in enumerate(design_list, 1):
                        print(f"  {i}. {design}")
                        
                    # 활성 디자인
                    active_design = desktop.active_design()
                    if active_design:
                        print(f"🎯 활성 디자인: {active_design.GetName()}")
                        print(f"   디자인 타입: {active_design.GetDesignType()}")
                except:
                    print("디자인 정보 가져오기 실패")
            else:
                print("🎯 활성 프로젝트: 없음")
                
        except Exception as e:
            print(f"프로젝트 정보 가져오기 실패: {e}")
            
    except Exception as e:
        print(f"❌ Desktop 상태 확인 중 오류: {e}")

def smart_aedt_connector():
    """
    스마트 AEDT 연결 함수 - 사용자 친화적 인터페이스
    
    Returns:
    --------
    Desktop : 연결된 Desktop 객체
    """
    print("🚀 스마트 AEDT 연결기를 시작합니다...")
    
    # Desktop 연결 시도
    desktop = get_desktop_connection()
    
    if desktop:
        # 연결 성공 시 상태 확인
        check_current_desktop_status(desktop)
        
        print("\n" + "=" * 60)
        print("🎉 AEDT Desktop 연결이 완료되었습니다!")
        print("💡 다음과 같이 사용할 수 있습니다:")
        print("=" * 60)
        print("# 프로젝트 열기:")
        print("# project = desktop.open_project(r'C:\\path\\to\\your\\project.aedt')")
        print("#")
        print("# Maxwell 객체 생성:")
        print("# m2d = ansys.aedt.core.Maxwell2d(project=desktop, new_desktop=False)")
        print("# m3d = ansys.aedt.core.Maxwell3d(project=desktop, new_desktop=False)")
        print("=" * 60)
        
        return desktop
    else:
        print("❌ AEDT Desktop 연결에 실패했습니다.")
        print("\n🔍 문제 해결 방법:")
        print("1. Ansys AEDT가 설치되어 있는지 확인")
        print("2. AEDT 라이선스가 사용 가능한지 확인")
        print("3. 수동으로 AEDT를 실행한 후 다시 시도")
        return None

# 간단한 사용 함수들
def quick_connect():
    """빠른 연결 - 기존 세션 우선"""
    return try_connect_to_existing_desktop()

def force_new_session():
    """강제로 새 세션 생성"""
    try:
        return Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=True,
            non_graphical=NG_MODE
        )
    except Exception as e:
        print(f"새 세션 생성 실패: {e}")
        return None


## Load Maxwell 2D Model

In [24]:
# e10 Model
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.

In [25]:
# aedt_file=r"D:\KDHe10\e10_example\e10_tutorial_ANSYSEM_2D.aedt"
# m2d = ansys.aedt.core.Maxwell2d(
#     project=aedt_file,
#     version=AEDT_VERSION,
#     new_desktop=False,
#     non_graphical=NG_MODE
# )

## Utility: 현재 실행중인 Maxwell2d 객체 자동 연결 함수

In [26]:
def get_running_maxwell2d(aedt_version="2025.2", non_graphical=False):
    """
    현재 실행 중인 AEDT Desktop 세션에 연결하여 활성 Maxwell 2D 디자인의 Maxwell2d 객체를 반환합니다.

    동작 순서:
    1) 기존 Desktop 세션에 연결 (새 세션 생성하지 않음)
    2) 활성 프로젝트/디자인 조회
    3) 활성 디자인이 Maxwell 2D가 아니면, 프로젝트의 디자인 목록에서 Maxwell 2D를 탐색
    4) 찾은 프로젝트/디자인 이름으로 Maxwell2d 객체 attach

    Parameters
    ----------
    aedt_version : str
        AEDT 버전 문자열 (예: "2025.2").
    non_graphical : bool
        비그래픽 모드 여부. 기존 실행 세션에 attach할 때는 보통 False 권장.

    Returns
    -------
    ansys.aedt.core.Maxwell2d or None
        연결된 Maxwell2d 객체. 찾지 못하면 None 반환.
    """
    try:
        from ansys.aedt.core import Desktop, Maxwell2d
    except Exception as e:
        print(f"❌ PyAEDT import 실패: {e}")
        return None

    desktop = None
    try:
        # 기존 세션에 붙기 (새 세션 X)
        desktop = Desktop(
            specified_version=aedt_version,
            new_desktop_session=False,
            non_graphical=non_graphical
        )
    except Exception as e:
        print(f"❌ 기존 Desktop 연결 실패: {e}")
        return None

    # 활성 프로젝트/디자인 가져오기
    try:
        active_proj = desktop.active_project()
        if not active_proj:
            projs = desktop.project_list()
            if not projs:
                print("❌ 열린 프로젝트가 없습니다.")
                return None
            # 첫 프로젝트 활성화
            proj_name = projs[0]
            active_proj = desktop.open_project(proj_name)
        else:
            proj_name = active_proj.GetName()
    except Exception as e:
        print(f"❌ 프로젝트 정보 획득 실패: {e}")
        return None

    # 활성 디자인 확인 → Maxwell 2D인지 확인
    try:
        active_design = desktop.active_design()
        design_name = None
        design_type = None
        if active_design:
            design_name = active_design.GetName()
            design_type = active_design.GetDesignType()

        if not active_design or (design_type and "Maxwell" not in design_type) or (design_type and "2D" not in design_type):
            # 프로젝트의 디자인 목록에서 Maxwell 2D 탐색
            try:
                design_list = active_proj.GetTopDesignList()
            except Exception:
                design_list = []
            maxwell2d_name = None
            for dn in design_list:
                try:
                    d = active_proj.SetActiveDesign(dn)
                    # SetActiveDesign 반환이 None일 수 있으므로 다시 active_design 가져오기
                    ad = desktop.active_design()
                    if ad and "Maxwell" in ad.GetDesignType() and "2D" in ad.GetDesignType():
                        maxwell2d_name = ad.GetName()
                        break
                except Exception:
                    continue
            if not maxwell2d_name:
                print("❌ Maxwell 2D 디자인을 찾지 못했습니다.")
                return None
            design_name = maxwell2d_name
    except Exception as e:
        print(f"❌ 디자인 정보 획득 실패: {e}")
        return None

    # Maxwell2d 객체 attach
    try:
        m2d_attached = Maxwell2d(
            project=proj_name,
            design=design_name,
            version=aedt_version,
            new_desktop=False,
            non_graphical=non_graphical
        )
        print(f"✅ 연결 성공: Project='{proj_name}', Design='{design_name}'")
        return m2d_attached
    except Exception as e:
        print(f"❌ Maxwell2d attach 실패: {e}")
        return None

In [27]:
# 사용 예시
print("="*70)
print("🔌 현재 실행 중 Maxwell2d 객체 가져오기")
print("="*70)

m2d_running = get_running_maxwell2d()
if m2d_running:
    print(f"Design Type: {m2d_running.design_type}")
    print(f"Variables: {list(m2d_running.variable_manager.variables.keys())[:5]} ...")
else:
    print("⚠️ Maxwell2d 객체를 가져오지 못했습니다.")

🔌 현재 실행 중 Maxwell2d 객체 가져오기
PyAEDT INFO: Python version 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT WARNING: Argument `specified_version` is deprecated for method `__init__`; use `version` instead.
PyAEDT WARNING: Argument `new_desktop_session` is deprecated for method `__init__`; use `new_desktop` instead.
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Python version 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT INFO: Project e10_DOE set to active.
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Aedt Objects correctly read
✅ 연결 성공: Project='e10_DOE', Design='Motor-CAD e10_tutorial'
Design Type: Maxwell 2D
Variables: ['Is_2D_Design', 'DiaGap', 'DiaStatorYoke', 'Dia

In [28]:
m2d=m2d_running

### Get Design Information

In [7]:
designName=m2d.design_list
display(designName)


['Motor-CAD e10_tutorial', 'Motor-CAD e10_tutorial_BPM_LabModel_1']

In [8]:
m2d.set_active_design(designName[0])

PyAEDT INFO: Python version 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT INFO: Project e10_tutorial_ANSYSEM_2D set to active.
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Aedt Objects correctly read


True

In [29]:
m2d.close_desktop()

PyAEDT INFO: Desktop has been released and closed.


True

## FLD Export Function Definition

In [9]:
m2dpost=m2d.post
all_objects = m2d.modeler.object_names

# Modelplotter=m2dpost.get_model_plotter_geometries(generate_mesh=True,get_objects_from_aedt=True)

PyAEDT INFO: Parsing F:\KDH\Thesis\JEET\e10_tuto\e10_tutorial_ANSYSEM_2D.aedt.
PyAEDT INFO: File F:\KDH\Thesis\JEET\e10_tuto\e10_tutorial_ANSYSEM_2D.aedt correctly loaded. Elapsed time: 0m 0sec
PyAEDT INFO: aedt file load time 0.5107967853546143
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 1sec
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Modeler2D class has been initialized!
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 1sec


### mesh export as *.case 

In [10]:
desktop = ansys.aedt.core.Desktop() 
pjtPath=desktop.project_path()
prjName=desktop.active_project().GetName()
filePath=os.path.join(pjtPath, prjName) 
pjt=desktop.load_project(filePath)
setup=pjt.get_setup(name='Setup1')
FieldReporter=pjt.get_module("FieldsReporter")


PyAEDT INFO: Python version 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Python version 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT INFO: Project e10_tutorial_ANSYSEM_2D set to active.
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Aedt Objects correctly read


In [11]:
SetupObj=m2d.get_setup('Setup1')
{"Time":SetupObj.props['MaxTimeStep']}
import ansys.aedt.core.visualization.plot.pyvista as AEDTvista
AEDTvista
import ansys.aedt.core.visualization.post as AEDTpost
AEDTpost

c:\Users\user\.ansys_python_venvs\pyAEDT_py311\Lib\site-packages\ansys\aedt\core\visualization\plot\pyvista.py:52: UserWarning: Graphics dependencies are required. Please install the ``graphics`` target to use this method. You can install it by running `pip install pyaedt[graphics]` or `pip install pyaedt[all]`.
  warnings.warn(ERROR_GRAPHICS_REQUIRED)


<module 'ansys.aedt.core.visualization.post' from 'c:\\Users\\user\\.ansys_python_venvs\\pyAEDT_py311\\Lib\\site-packages\\ansys\\aedt\\core\\visualization\\post\\__init__.py'>

## plot Field

In [12]:
m2dpost.available_quantities_categories()

['Torque',
 'Speed',
 'Position',
 'Winding',
 'Loss',
 'Misc. Solution',
 'Demag Percentage',
 'Design',
 'Expression Cache',
 'Expression Converge']

In [ ]:
m2dpost.get_solution_data

In [33]:
solutions = m2d.post.get_solution_data(
    primary_sweep_variable="Time", domain="Sweep"
)


PyAEDT WARNING: No report category provided. Automatically identified Transient
PyAEDT INFO: Solution Data Correctly Loaded.
Time to initialize solution data:0.33446478843688965
Time to initialize solution data:0.34018945693969727


In [22]:
timeSteps=solutions.variation_values(variation='Time')

NameError: name 'solutions' is not defined

In [47]:
timeUnit=solutions.units_sweeps['Time']

In [ ]:
str(timeSteps[0])+timeUnit

'0.0ns'

In [ ]:
for time in timeSteps:
    plot=m2dpost.plot_field(
        quantity="",
        assignment=all_objects,
        plot_type="Surface",
        show=False,
        mesh_on_fields=True,
        file_format="case",
        plot_cad_objs=True,
        intrinsics={"Time":str(time)+timeUnit}
    )

PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set 

In [53]:
plot=m2dpost.plot_field(
    quantity="Mag_B",
    assignment=all_objects,
    plot_type="Surface",
    show=False,
    mesh_on_fields=True,
    file_format="case",
    intrinsics={"Time":"0.000344827586206896s"}
)

PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial


In [13]:
import pyvista as pv

# 인터랙티브 렌더링 설정
pv.set_jupyter_backend('trame')  # 'panel'이 가장 인터랙티브함

case_file = r"F:\KDH\Thesis\JEET\e10_tuto\e10_tutorial_ANSYSEM_2D.pyaedt\Motor-CAD_e10_tutorial\Mag_B_KXZ0HU.case"
reader = pv.get_reader(case_file)
mesh = reader.read()

print(f"📦 데이터 타입: {type(mesh).__name__}")

if isinstance(mesh, pv.MultiBlock):
    print(f"🔹 MultiBlock: {mesh.n_blocks}개 블록")
    
    # 인터랙티브 Plotter 생성
    plotter = pv.Plotter(notebook=True)
    
    # 각 블록 추가
    for i, block in enumerate(mesh):
        if block is not None and block.n_points > 0:
            # 첫 번째 유효한 필드 찾기
            if block.point_data:
                field_name = list(block.point_data.keys())[0]
                print(f"블록 {i}: {field_name} 필드 사용")
                
                # 필드 컬러맵으로 표시
                plotter.add_mesh(
                    block,
                    scalars=field_name,
                    show_edges=True,  # ✨ Mesh 표시
                    edge_color='black',  # Mesh 선 색상
                    line_width=0.5,  # Mesh 선 두께
                    cmap='jet',
                    opacity=1.0,
                    scalar_bar_args={
                        'title': field_name,
                        'vertical': True,
                        'height': 0.25,
                        'width': 0.05,
                        'position_x': 0.85,
                        'position_y': 0.05
                    }
                )
    
    # 카메라 및 조명 설정
    plotter.enable_anti_aliasing()
    plotter.add_axes()
    
    # 인터랙티브 Plot 표시
    plotter.show(jupyter_backend='trame')
    
else:
    # 단일 메시
    print(f"📊 Points: {mesh.n_points}")
    print(f"📐 Cells: {mesh.n_cells}")
    
    if mesh.point_data:
        field_name = list(mesh.point_data.keys())[0]
        
        plotter = pv.Plotter(notebook=True)
        plotter.add_mesh(
            mesh,
            scalars=field_name,
            show_edges=True,  # ✨ Mesh 표시
            edge_color='black',
            line_width=0.5,
            cmap='jet',
            scalar_bar_args={'title': field_name}
        )
        plotter.enable_anti_aliasing()
        plotter.add_axes()
        plotter.show(jupyter_backend='trame')

📦 데이터 타입: MultiBlock
🔹 MultiBlock: 1개 블록
블록 0: Mag_B 필드 사용


Widget(value='<iframe src="http://localhost:54082/index.html?ui=P_0x1883231a990_0&reconnect=auto" class="pyvis…

In [ ]:

gif = m2d.post.plot_animated_field(
    quantity="Mag_B",
    assignment=all_objects,
    plot_type="Surface",
    intrinsics={"Time": "0s"},
    variation_variable="Time",
    variations=timesteps,
    show=False,
    export_gif=False,
)
gif.isometric_view = False
gif.camera_position = [15, 15, 80]
gif.focal_point = [15, 15, 0]
gif.roll_angle = 0
gif.elevation_angle = 0
gif.azimuth_angle = 0

# Set off_screen to False to visualize the animation.
# gif.off_screen = False
gif.animate()

## From Hori san


In [ ]:

sWorkingFolder = r'D:\SingleFieldExportFromMaxwell\_2025R2_Test\Case1_2025R1'
#sWorkingFolder = r'D:\SingleFieldExportFromMaxwell\_2025R2_Test\Case2'
sTargetObjName =  'OL_HopperAir'
sFieldQuantity = 'B'    #B, E, D, or H
sSetupName = 'Setup1'
sVersion = ''           # empty means current AEDT ver.
#sVersion = '2024.1'    # Enforce to sexport 2024R1 format



import os, time
from System.IO import File
import ScriptEnv
ScriptEnv.Initialize("Ansoft.ElectronicsDesktop")
oDesktop.RestoreWindow()
oProject = oDesktop.GetActiveProject()
oDesign = oProject.GetActiveDesign()
oModule = oDesign.GetModule("FieldsReporter")
if sVersion == '':
    aVer = oDesktop.GetVersion().strip().split('.')
else:
    aVer = sVersion.strip().split('.')
    
aComp = ["X", "Y", "Z"]

#tStart = time.time()
#oDesktop.AddMessage(oProject.GetName(), oDesign.GetName(), 0, 'Export: ' + str(tStart), "")

oModule.EnterQty(sFieldQuantity)
oModule.CalcOp("Smooth")
oModule.EnterVol(sTargetObjName)
oModule.CalcOp("Value")
oModule.CalculatorWrite(os.path.join(sWorkingFolder, sFieldQuantity + 'vec.fld'), 
        ["Solution:=", sSetupName + " : LastAdaptive"], [])

for i in range (3):
    oModule.EnterQty(sFieldQuantity)
    #oModule.CalcOp("Smooth")
    oModule.CalcOp("Scalar" + aComp[i])
    oModule.CalcOp("Grad")
    oModule.CalcOp("Smooth")
    oModule.EnterVol(sTargetObjName)
    oModule.CalcOp("Value")
    oModule.CalculatorWrite(os.path.join(sWorkingFolder, sFieldQuantity + 'grad' + aComp[i] + '.fld'), 
        ["Solution:=", sSetupName + " : LastAdaptive"], [])

#tWrite = time.time()
#oDesktop.AddMessage(oProject.GetName(), oDesign.GetName(), 0, 'Export: ' + str(tWrite), "")

fDiv = []
fvec = open(os.path.join(sWorkingFolder, sFieldQuantity + 'vec.fld'),"r")
for i in range (3):
    fDiv.append(open(os.path.join(sWorkingFolder, sFieldQuantity + 'grad' + aComp[i] + '.fld'),"r"))

# Skip 1st and 2nd line
line = fvec.readline()
line = fvec.readline()
for i in range (3):
    line = fDiv[i].readline()
    line = fDiv[i].readline()

# Read data4
allVec = fvec.read().strip().split('\n')
aPointCoord = list(range(len(allVec)))
aData = list(range(len(allVec)))

allGrad = list(range(3))
for i in range (3):
    allGrad[i] = fDiv[i].read().strip().split('\n')

for i, line in enumerate(allVec):
    temp = line.strip().split()
    aPointCoord[i] = (','.join(temp[:3]))
    str = ' '.join(temp)
    for j in range (3):
        temp = allGrad[j][i].strip().split()
        str = str + ' ' + ' '.join(temp[3:])
    aData[i] = str + '\n'

# Duplicated data check
da = {}
for i, key in enumerate(aPointCoord):
    da[key] = aData[i]

fres = open(os.path.join(sWorkingFolder, sFieldQuantity + '_vec.txt'),"w")

if int(aVer[0]) >= 2025 and int(aVer[1]) >= 2:
    fres.writelines('x y z %s:x %s:y %s:z d%sx:x d%sx:y d%sx:z d%sy:x d%sy:y d%sy:z d%sz:x d%sz:y d%sz:z\n'%(sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity))
    #print('x y z %sx %sy %sz d%sx:x d%sx:y d%sx:z d%sy:x d%sy:y d%sy:z d%sz:x d%sz:y d%sz:z\n'%(sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity))
elif int(aVer[0]) == 2025 and int(aVer[1]) == 1:
    fres.writelines('X Y Z %sx %sy %sz d%sx_dx d%sx_dy d%sx_dz d%sy_dx d%sy_dy d%sy_dz d%sz_dx d%sz_dy d%sz_dz\n'%(sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity))
    #print('X Y Z %sx %sy %sz d%sx_dx d%sx_dy d%sx_dz d%sy_dx d%sy_dy d%sy_dz d%sz_dx d%sz_dy d%sz_dz\n'%(sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity))
else:
    fres.writelines('X Y Z %sx %sy %sz d%sxdx d%sxdy d%sxdz d%sydx d%sydy d%sydz d%szdx d%szdy d%szdz\n'%(sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity))
    #print('X Y Z %sx %sy %sz d%sxdx d%sxdy d%sxdz d%sydx d%sydy d%sydz d%szdx d%szdy d%szdz\n'%(sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity, sFieldQuantity))

for value in da.values():
    fres.writelines(value)

fvec.close()
fres.close()
for i in range (3):
    fDiv[i].close()
